# GPU

## Execute with only 1 model

In [7]:
import onnxruntime as ort

print("ONNXRuntime device:", ort.get_device())
print("Available providers:", ort.get_available_providers())


ONNXRuntime device: GPU
Available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [6]:
from pathlib import Path
import onnxruntime as ort
from rembg import remove, new_session

# ✅ Auto-detect "images" folder (works from /code or project root)
candidates = [
    Path("images"),          # if notebook runs from project root
    Path("..") / "images",   # if notebook runs from /code
]

INPUT_DIR = None
for c in candidates:
    if c.exists():
        INPUT_DIR = c.resolve()
        break

if INPUT_DIR is None:
    raise FileNotFoundError("❌ Could not find the 'images' folder. Check your working directory.")

print("✅ Using INPUT_DIR:", INPUT_DIR)

# ✅ Choose ONE model
MODEL_NAME = "isnet-general-use"

# ✅ Output folder
OUTPUT_DIR = INPUT_DIR / f"output_GPU_{MODEL_NAME}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ✅ Providers (GPU first, CPU fallback)
available = ort.get_available_providers()
if "CUDAExecutionProvider" in available:
    PROVIDERS = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    print("✅ CUDA available → Using GPU")
else:
    PROVIDERS = ["CPUExecutionProvider"]
    print("⚠️ CUDA NOT available → Using CPU only")

print("Providers:", PROVIDERS)

# ✅ Create session (GPU if available)
session = new_session(MODEL_NAME, providers=PROVIDERS)

# ✅ Collect images
extensions = ["*.png", "*.jpg", "*.jpeg", "*.webp"]
files = []
for ext in extensions:
    files.extend(INPUT_DIR.glob(ext))

files = sorted(files)

print(f"✅ Model: {MODEL_NAME}")
print(f"✅ Found {len(files)} images")
print(f"✅ Output: {OUTPUT_DIR}")

if not files:
    raise SystemExit("❌ No images found in input folder.")

# ✅ Process images
for file in files:
    out_path = OUTPUT_DIR / f"{file.stem}.png"  # always PNG output

    with open(file, "rb") as i:
        input_data = i.read()

    output_data = remove(input_data, session=session)

    with open(out_path, "wb") as o:
        o.write(output_data)

    print(f"✅ {file.name} -> {out_path.name}")

print("\n🎉 Done (ONE model)!")


✅ Using INPUT_DIR: C:\Users\ignac\OneDrive\Desktop\rembg\images
✅ CUDA available → Using GPU
Providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
✅ Model: isnet-general-use
✅ Found 1 images
✅ Output: C:\Users\ignac\OneDrive\Desktop\rembg\images\output_GPU_isnet-general-use
✅ 1.jpeg -> 1.png

🎉 Done (ONE model)!


## Execute with all models

In [8]:
from pathlib import Path
import onnxruntime as ort
from rembg import remove, new_session

MODELS = [
    "u2net",
    "u2netp",
    "u2net_human_seg",
    "u2net_cloth_seg",
    "silueta",
    "isnet-general-use",
    "isnet-anime",
    "sam",
    "birefnet-general",
    "birefnet-general-lite",
    "birefnet-portrait",
    "birefnet-dis",
    "birefnet-hrsod",
    "birefnet-cod",
    "birefnet-massive",
    "bria-rmbg",
]

# ✅ Auto-detect "images" folder (works from /code or project root)
candidates = [
    Path("images"),          # if notebook runs from project root
    Path("..") / "images",   # if notebook runs from /code
]

INPUT_DIR = None
for c in candidates:
    if c.exists():
        INPUT_DIR = c.resolve()
        break

if INPUT_DIR is None:
    raise FileNotFoundError("❌ Could not find the 'images' folder. Check your working directory.")

print("✅ Using INPUT_DIR:", INPUT_DIR)

# ✅ Master output folder
BASE_OUTPUT_DIR = INPUT_DIR / "output_all_models_GPU"
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ✅ Providers (GPU first, CPU fallback)
available = ort.get_available_providers()
if "CUDAExecutionProvider" in available:
    PROVIDERS = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    print("✅ CUDA available → Using GPU")
else:
    PROVIDERS = ["CPUExecutionProvider"]
    print("⚠️ CUDA NOT available → Using CPU only")

print("Providers:", PROVIDERS)

# ✅ Collect images
extensions = ["*.png", "*.jpg", "*.jpeg", "*.webp"]
files = []
for ext in extensions:
    files.extend(INPUT_DIR.glob(ext))

files = sorted(files)

print(f"\n✅ Found {len(files)} images")
print(f"✅ Output base folder: {BASE_OUTPUT_DIR}")

if not files:
    raise SystemExit("❌ No images found in input folder.")

# ✅ Run each model
for model_name in MODELS:
    print("\n" + "=" * 60)
    print(f"🧠 Running model: {model_name}")

    model_output_dir = BASE_OUTPUT_DIR / model_name
    model_output_dir.mkdir(parents=True, exist_ok=True)

    try:
        session = new_session(model_name, providers=PROVIDERS)
    except Exception as e:
        print(f"❌ Could not initialize model '{model_name}'. Skipping. Error: {e}")
        continue

    for file in files:
        output_path = model_output_dir / f"{file.stem}.png"

        try:
            with open(file, "rb") as i:
                input_data = i.read()

            output_data = remove(input_data, session=session)

            with open(output_path, "wb") as o:
                o.write(output_data)

            print(f"✅ {file.name} -> {output_path.name}")

        except Exception as e:
            print(f"❌ Failed on {file.name} with model '{model_name}'. Error: {e}")

print("\n🎉 Finished processing ALL models (GPU if available)!")

✅ Using INPUT_DIR: C:\Users\ignac\OneDrive\Desktop\rembg\images
✅ CUDA available → Using GPU
Providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']

✅ Found 1 images
✅ Output base folder: C:\Users\ignac\OneDrive\Desktop\rembg\images\output_all_models_GPU

🧠 Running model: u2net
✅ 1.jpeg -> 1.png

🧠 Running model: u2netp
✅ 1.jpeg -> 1.png

🧠 Running model: u2net_human_seg
✅ 1.jpeg -> 1.png

🧠 Running model: u2net_cloth_seg
✅ 1.jpeg -> 1.png

🧠 Running model: silueta
✅ 1.jpeg -> 1.png

🧠 Running model: isnet-general-use
✅ 1.jpeg -> 1.png

🧠 Running model: isnet-anime
✅ 1.jpeg -> 1.png

🧠 Running model: sam
✅ 1.jpeg -> 1.png

🧠 Running model: birefnet-general
✅ 1.jpeg -> 1.png

🧠 Running model: birefnet-general-lite
✅ 1.jpeg -> 1.png

🧠 Running model: birefnet-portrait
✅ 1.jpeg -> 1.png

🧠 Running model: birefnet-dis
✅ 1.jpeg -> 1.png

🧠 Running model: birefnet-hrsod
✅ 1.jpeg -> 1.png

🧠 Running model: birefnet-cod
✅ 1.jpeg -> 1.png

🧠 Running model: birefnet-massive
✅ 1.jpeg -> 1.p